# 02 — End-to-End RAG Retrieval Pipeline

This notebook implements the full **Retrieve → Rerank → Summarize** pipeline.

**Prerequisites:** Run `01_build_index.ipynb` first to generate the FAISS index and embeddings.

**Pipeline stages:**
1. **Retrieve** — Encode query with MiniLM, search FAISS index for top-50 candidates
2. **Rerank** — Score each (query, tweet) pair with a cross-encoder; keep top-10
3. **Summarize** — Feed top-5 tweets into FLAN-T5 to generate a natural language summary

This two-stage approach balances speed and accuracy: FAISS retrieves fast candidates, the cross-encoder applies a more expensive but more accurate relevance model.

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
# transformers: for FLAN-T5
# sentence-transformers: for MiniLM query encoder and cross-encoder reranker
# faiss-cpu: for loading the vector index
!pip install transformers sentence-transformers faiss-cpu torch --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# --- STEP 2: LOAD INDEX AND TWEET LIST ---
# These files were created by 01_build_index.ipynb

import os

# Support both Colab (/content/) and local environments
base = '/content/' if os.path.exists('/content/minilm_faiss.index') else ''

print("Loading FAISS index...")
minilm_index = faiss.read_index(f'{base}minilm_faiss.index')
print(f"Index loaded — contains {minilm_index.ntotal:,} vectors")

print("Loading tweet texts...")
with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)
print(f"Loaded {len(tweets):,} tweet texts")

In [ ]:
# --- STEP 3: LOAD MODELS ---

print("Loading MiniLM query encoder...")
# We use the same model as the index was built with — mismatching models would give garbage results
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print("MiniLM loaded.")

print("Loading cross-encoder reranker...")
# The cross-encoder reads (query, passage) together — much more accurate than bi-encoder
# but too slow to run on 110k tweets directly, so we only apply it to the top-50 candidates
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-encoder loaded.")

print("Loading FLAN-T5 for summarization...")
# FLAN-T5 is an instruction-tuned encoder-decoder model — it follows natural language prompts
# well and generates coherent summaries without any fine-tuning on our data.
# We prefer 'large' for better summary quality; fall back to 'base' if OOM.
try:
    flan_model_name = 'google/flan-t5-large'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_model_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_model_name).to(device)
    print(f"FLAN-T5-large loaded on {device}.")
except RuntimeError:
    # flan-t5-large needs ~3GB RAM — fall back to base (~900MB) on memory-constrained environments
    print("OOM on flan-t5-large — falling back to flan-t5-base (smaller but still instruction-tuned).")
    flan_model_name = 'google/flan-t5-base'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_model_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_model_name).to(device)
    print(f"FLAN-T5-base loaded on {device}.")

In [ ]:
# --- STEP 4: RETRIEVAL FUNCTION ---

def retrieve(query, top_k=50):
    """
    Encode a query with MiniLM and retrieve the top-k most similar tweets
    from the FAISS index using cosine similarity (inner product on normalized vectors).

    Parameters:
        query (str): The search query.
        top_k (int): Number of candidate tweets to retrieve. Default 50
                     gives the reranker enough candidates to work with.

    Returns:
        list[tuple[int, float]]: List of (tweet_index, similarity_score) pairs,
                                  ordered from most to least similar.
    """
    query_vec = bi_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    # Normalize so inner product equals cosine similarity — mirrors how the index was built
    query_vec = query_vec / np.linalg.norm(query_vec)

    scores, indices = minilm_index.search(query_vec, top_k)
    return list(zip(indices[0].tolist(), scores[0].tolist()))


print("retrieve() function defined.")

In [ ]:
# --- STEP 5: RERANKING FUNCTION ---

def rerank(query, candidates, top_n=10):
    """
    Rerank a list of candidate tweets using a cross-encoder model.

    Unlike bi-encoders (which encode query and tweet independently),
    a cross-encoder reads both together and produces a single relevance score.
    This is more accurate but too slow to run on 110k tweets — hence the
    two-stage approach: FAISS narrows to ~50, cross-encoder picks the best 10.

    Parameters:
        query (str): The original search query.
        candidates (list[tuple[int, float]]): Output of retrieve() — (index, score) pairs.
        top_n (int): How many to keep after reranking.

    Returns:
        list[tuple[int, float]]: Top-n (tweet_index, cross_encoder_score) pairs,
                                  ordered by reranked relevance.
    """
    tweet_indices = [idx for idx, _ in candidates]
    candidate_texts = [tweets[idx] for idx in tweet_indices]

    # CrossEncoder expects a list of [query, passage] pairs
    pairs = [[query, text] for text in candidate_texts]
    cross_scores = cross_encoder.predict(pairs)

    # Sort by cross-encoder score descending and return top_n
    scored = sorted(zip(tweet_indices, cross_scores), key=lambda x: x[1], reverse=True)
    return scored[:top_n]


print("rerank() function defined.")

In [ ]:
# --- STEP 6: SUMMARIZATION FUNCTION ---

def summarize(query, top_tweets, max_new_tokens=200):
    """
    Use FLAN-T5 to generate a natural language summary of the retrieved tweets.

    FLAN-T5 is encoder-decoder and instruction-tuned, so it responds well
    to prompts that tell it what to do. We pass the top retrieved tweets
    as context and ask it to summarize public opinion on the query topic.

    Parameters:
        query (str): The original search query (used in the prompt).
        top_tweets (list[tuple[int, float]]): Output of rerank() — (index, score) pairs.
                                               We use the top 5 for the summary context.
        max_new_tokens (int): Maximum length of the generated summary.

    Returns:
        str: A natural language summary of what people are saying about the query.
    """
    # Use at most 5 tweets as context to keep the prompt within T5's token limit
    context_tweets = [tweets[idx] for idx, _ in top_tweets[:5]]

    numbered = '\n'.join(f"{i+1}. {t}" for i, t in enumerate(context_tweets))
    prompt = (
        f"Context tweets:\n{numbered}\n\n"
        f"Based on the tweets above, summarize what people are saying about: {query}"
    )

    inputs = flan_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    # num_beams=4: beam search produces more coherent output than greedy decoding
    output_ids = flan_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    summary = flan_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary


print("summarize() function defined.")

In [ ]:
# --- STEP 7: FULL PIPELINE ---

def rag_search(query, retrieve_k=50, rerank_n=10):
    """
    Run the full end-to-end RAG pipeline: retrieve → rerank → summarize.

    Parameters:
        query (str): The natural language search query.
        retrieve_k (int): Number of candidates to pull from FAISS.
        rerank_n (int): Number of results after cross-encoder reranking.

    Returns:
        dict with keys:
            'query'     — original query string
            'retrieved' — list of (tweet_text, bi_encoder_score) before reranking
            'reranked'  — list of (tweet_text, cross_encoder_score) after reranking
            'summary'   — FLAN-T5 generated summary
    """
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")

    print(f"[1/3] Retrieving top-{retrieve_k} candidates from FAISS...")
    candidates = retrieve(query, top_k=retrieve_k)

    print(f"[2/3] Reranking with cross-encoder — keeping top-{rerank_n}...")
    reranked = rerank(query, candidates, top_n=rerank_n)

    print(f"[3/3] Summarizing top-5 with FLAN-T5...")
    summary = summarize(query, reranked)

    return {
        'query': query,
        'retrieved': [(tweets[idx], score) for idx, score in candidates[:10]],
        'reranked': [(tweets[idx], float(score)) for idx, score in reranked],
        'summary': summary
    }


print("rag_search() pipeline function defined.")

In [ ]:
# --- STEP 8: DEMO RUNS ---
# Run the full pipeline on a few example queries to see it in action.

demo_queries = [
    "I am looking for a job.",
    "climate change and the environment",
    "feeling happy and excited today"
]

results = []
for q in demo_queries:
    result = rag_search(q)
    results.append(result)

    print(f"\nTop 5 reranked tweets:")
    for i, (text, score) in enumerate(result['reranked'][:5], start=1):
        print(f"  {i}. [score={score:.3f}] {text[:120]}")

    print(f"\nFLAN-T5 Summary:")
    print(f"  {result['summary']}")
    print()